# Script para geração do CSV final

In [4]:
# Installs (descomente se precisar)
# !pip install -q openpyxl

import pandas as pd
import numpy as np

# caminho do arquivo
arquivo = "../datasets/PCOS_data_without_infertility.xlsx"
sheet = "Full_new"

# carregar
df = pd.read_excel(arquivo, sheet_name=sheet)

# padronizar nomes de colunas (remover espaços nas pontas)
df.columns = df.columns.str.strip()

# remover colunas irrelevantes/fantasma e IDs
colunas_para_remover = ["Sl. No", "Patient File No.", "Unnamed: 44"]
df = df.drop(columns=[c for c in colunas_para_remover if c in df.columns], errors="ignore")

# converter quaisquer colunas lidas como object que deveriam ser numéricas
#    (convertendo tudo que for possível; valores inválidos virarão NaN)
obj_cols = df.select_dtypes(include="object").columns.tolist()
for c in obj_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# checar nulos antes da imputação (opcional - apenas para log)
nulos_antes = df.isna().sum()
print("Nulos antes da imputação (colunas com >0):")
print(nulos_antes[nulos_antes > 0])

# imputação: preencher valores ausentes com a mediana da coluna
#    - usa mediana para variáveis numéricas
#    - para colunas que ficaram não-numéricas (se houver), preenche com modo
for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            med = df[col].median()
            df[col] = df[col].fillna(med)
            print(f"Preenchido NaNs em '{col}' com mediana = {med}")
        else:
            # fallback para colunas categóricas (poucas no dataset esperado)
            modo = df[col].mode(dropna=True)
            fill = modo.iloc[0] if not modo.empty else ""
            df[col] = df[col].fillna(fill)
            print(f"Preenchido NaNs em '{col}' com modo = {fill}")

# garantir que todas as colunas estão numéricas
#    se houver colunas não-numéricas remanescentes, tenta cast para float; se falhar, remove.
nonnumeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
if nonnumeric:
    print("Colunas não numéricas remanescentes (serão convertidas quando possível):", nonnumeric)
    for c in nonnumeric:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    # se ainda houver NaNs depois da conversão, preencher com mediana
    for c in nonnumeric:
        if df[c].isna().any():
            med = df[c].median()
            df[c] = df[c].fillna(med)
            print(f"Depois da conversão, preenchido '{c}' com mediana = {med}")

# checagem final: sem nulos e todas numéricas
total_nulos = df.isna().sum().sum()
print(f"Total de nulos restantes: {total_nulos}")
assert total_nulos == 0, "Ainda existem valores nulos — reveja os passos anteriores."

# (opcional) reset index - importante para tratamento do pandas - exportar CSV final pronto para modelagem
df = df.reset_index(drop=True)
saida = "../datasets/pcos_clean.csv"
df.to_csv(saida, index=False)
print(f"Arquivo salvo: {saida} — shape final: {df.shape[0]} linhas × {df.shape[1]} colunas")

Nulos antes da imputação (colunas com >0):
Marraige Status (Yrs)     1
II    beta-HCG(mIU/mL)    1
AMH(ng/mL)                1
Fast food (Y/N)           1
dtype: int64
Preenchido NaNs em 'Marraige Status (Yrs)' com mediana = 7.0
Preenchido NaNs em 'II    beta-HCG(mIU/mL)' com mediana = 1.99
Preenchido NaNs em 'AMH(ng/mL)' com mediana = 3.7
Preenchido NaNs em 'Fast food (Y/N)' com mediana = 1.0
Total de nulos restantes: 0
Arquivo salvo: ../datasets/pcos_clean.csv — shape final: 541 linhas × 42 colunas
